<div dir="rtl">

# تمرين 4.2: الأداة الضعيفة

**برنامج التجارب واختبارات A/B والاستدلال السببي**، اليوم الرابع

**الخطوات:**
1. من قائمة File اختر Save a copy in Drive، واعمل على النسخة التي تُفتح.
2. شغّل الخلايا بالترتيب، ثم اكتب حلك في الخلية المخصصة.
3. اضغط Share، واجعل General access على Anyone with the link بصلاحية Viewer، ثم اضغط Copy link.
4. الصق الرابط في ملف التقييم الخاص بمجموعتك، في عمود «تمرين 4.2».

</div>

<div dir="rtl">

## الإعداد

شغّل الخلية التالية أولًا.

</div>

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import html
from IPython.display import HTML, display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler
from matplotlib import ft2font
from scipy import stats

# الإصدارات الحديثة من matplotlib تعرض العربية مباشرة، والقديمة تحتاج مكتبتين صغيرتين
NATIVE_ARABIC = hasattr(ft2font, "__libraqm_version__")
if not NATIVE_ARABIC:
    try:
        import arabic_reshaper
        from bidi.algorithm import get_display
    except ImportError:
        import subprocess
        import sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "arabic-reshaper", "python-bidi"], check=True)
        import arabic_reshaper
        from bidi.algorithm import get_display

rng = np.random.default_rng(2026)

# ألوان البرنامج: A تركوازي، وB برتقالي، والقيمة الحقيقية كحلي، والطريقة الساذجة رمادي
TEAL, ORANGE, NAVY, GRAY = "#0E7490", "#E58A1F", "#0F2A3F", "#A9B4BC"
plt.rcParams.update({
    "figure.figsize": (9, 4.5), "figure.dpi": 110,
    "axes.prop_cycle": cycler(color=[TEAL, ORANGE, NAVY, GRAY]),
    "axes.edgecolor": "#56656F", "axes.labelcolor": NAVY, "text.color": NAVY,
    "xtick.color": "#3D4F5C", "ytick.color": "#3D4F5C",
    "axes.grid": True, "grid.color": "#D8E1E7", "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 13, "axes.titleweight": "bold", "font.size": 11,
    "legend.frameon": False,
})
pd.set_option("display.precision", 4)


def ar(text):
    """يجهز النص العربي ليظهر بالشكل والاتجاه الصحيحين داخل الرسوم."""
    text = str(text)
    return text if NATIVE_ARABIC else get_display(arabic_reshaper.reshape(text))


def say(*parts, sep=" "):
    """يطبع نصًا عربيًا من اليمين إلى اليسار."""
    text = html.escape(sep.join(str(p) for p in parts))
    display(HTML(f'<div dir="rtl" style="white-space: pre-wrap; line-height: 1.7;">{text}</div>'))


import networkx as nx
import statsmodels.formula.api as smf
from IPython.display import Markdown, display


def tidy(fit, terms=None):
    """جدول مختصر لنتيجة الانحدار."""
    ci = fit.conf_int()
    out = pd.DataFrame({"المعامل": fit.params, "SE": fit.bse, "أدنى 95%": ci[0], "أعلى 95%": ci[1], "p": fit.pvalues})
    return out.loc[terms] if terms else out


def draw_dag(ax, edges, pos, title, highlight=None, unobserved=()):
    """يرسم رسمًا سببيًا: البرتقالي هو المتغير المهم في المثال، والرمادي متغير لا نقيسه."""
    g = nx.DiGraph(edges)
    colors = ["#FCEBD7" if v == highlight else ("#E4E9ED" if v in unobserved else "#E0F0F4") for v in g.nodes]
    nx.draw_networkx(g, pos=pos, ax=ax, labels={v: ar(v) for v in g.nodes}, node_color=colors, node_size=4600,
                     font_size=9, font_color=NAVY, edgecolors=NAVY, edge_color=NAVY, arrowsize=16, width=1.2)
    ax.set_title(ar(title), fontsize=10.5)
    ax.axis("off")
    ax.margins(0.25)


say("الإعداد جاهز")

<div dir="rtl">

## الأدوات الجاهزة

هذه الدوال والبيانات من دفتر اليوم، ويحتاجها التمرين. شغّلها كما هي.

</div>

In [ ]:
def simulate_iv(n=20_000, strength=1.5, seed=3):
    r = np.random.default_rng(seed)
    motivation = r.normal(size=n)                                         # U: الحماس، ولا نقيسه
    sms = r.binomial(1, 0.5, n)                                           # Z: رسالة تذكير بالقرعة
    uses = r.binomial(1, 1 / (1 + np.exp(-(-1.0 + strength * sms + 1.2 * motivation))))   # X: استخدام الميزة
    spend = 50 + 10 * uses + 15 * motivation + r.normal(0, 10, n)         # Y: الإنفاق، والأثر الحقيقي 10
    return pd.DataFrame({"sms": sms, "uses": uses, "spend": spend})


def wald(df):
    """أثر الأداة على النتيجة مقسومًا على أثر الأداة على المعالجة."""
    z1, z0 = df["sms"] == 1, df["sms"] == 0
    return ((df.loc[z1, "spend"].mean() - df.loc[z0, "spend"].mean())
            / (df.loc[z1, "uses"].mean() - df.loc[z0, "uses"].mean()))


iv = simulate_iv()
first_stage = smf.ols("uses ~ sms", iv).fit()
ols_biased = smf.ols("spend ~ uses", iv).fit().params["uses"]
say(f"المرحلة الأولى: الرسالة رفعت الاستخدام {first_stage.params['sms'] * 100:.1f} نقطة، وقيمة F = {first_stage.fvalue:,.0f}")
say(f"الانحدار العادي (متحيز بسبب الحماس): {ols_biased:.2f}")
say(f"تقدير الأداة (Wald): {wald(iv):.2f}، والأثر الحقيقي 10")

<div dir="rtl">

## المطلوب

**الأداة الضعيفة:** كرر التقدير على 200 مجموعة بيانات محاكاة (n = 5,000)، مرة بأداة قوية (`strength=1.5`) ومرة بأداة ضعيفة (`strength=0.1`). قارن قيمة F وانتشار التقديرات. ماذا تستنتج؟

</div>

In [ ]:
# اكتب حلك هنا
# لكل قيمة strength في [1.5, 0.1]: كرر simulate_iv(n=5_000, strength=..., seed=1_000 + s) ثم wald(...)

rows = []
for strength in [1.5, 0.1]:
    for s in range(200):
        d = simulate_iv(n=5_000, strength=strength, seed=1_000+ s)
        rows.append({"القوة": strength,
                     "F": smf.ols("uses ~ sms", d).fit().fvalue,
                     "التقدير": wald(d)})
sims = pd.DataFrame(rows)

summary = sims.groupby("القوة").agg(
    **{"وسيط F": ("F", "median"),
       "نسبة F < 10": ("F", lambda x: (x < 10).mean()),
       "وسيط التقدير": ("التقدير", "median"),
       "الانحراف المعياري": ("التقدير", "std"),
       "المئين 5": ("التقدير", lambda x: x.quantile(0.05)),
       "المئين 95": ("التقدير", lambda x: x.quantile(0.95))})

summary.round(2)


,وسيط F,نسبة F < 10,وسيط التقدير,الانحراف المعياري,المئين 5,المئين 95
القوة,,,,,,
0.1,1.65,0.96,14.26,356.52,-151.85,104.29
1.5,430.35,0.00,10.04,1.84,6.72,12.89


<div dir="rtl">

## التسليم

انتهيت؟ اضغط Share، ثم Copy link، والصق الرابط في ملف التقييم في عمود «تمرين 4.2».

</div>